# Per-species spin-orbit coupling scaling

Upstream Elk only has a single global spin-orbit coupling (SOC) scale, `socscf`
(elk.in, manual sec. 5.129): the Koelling-Harmon SOC term added to the second-
variational Hamiltonian is multiplied by one scalar for the whole cell. That's the
wrong shape for a multi-species cell where SOC strength should vary by element
(SOC grows roughly with atomic number).

`Calculation(spinorb=True, soc_scale={"Bi": 1.5})` adds per-species control: a new
`elkpy_socscfsp(maxspecies)` array (`modmain.f90`) and `elkpy_socscale` input block
(`readinput.f90`), read inside `gensocfr.f90`'s existing per-atom loop, overriding
`socscf` per species. See `patches/0001-per-species-soc-scale.patch`,
`docs/design.md` #12, and `docs/physics.tex` Part I for the physics.

In [1]:
from elkpy.structure import Structure

## Reproducing the global `socscf` scalar

For a single-species cell, a per-species override on that one species must exactly
reproduce the effect of the global `socscf` scalar -- both paths go through the same
`gensocfr.f90` computation, just with a different scale value selected. Bismuth (Z =
83) has strong SOC, so the two energies should agree to high precision, not just
"roughly the same".

In [2]:
BI_AVEC = [(0.0, 5.0, 5.0), (5.0, 0.0, 5.0), (5.0, 5.0, 0.0)]
bi_structure = Structure(BI_AVEC, {"Bi": [(0.0, 0.0, 0.0)]})

e_global = bi_structure.get_calculation(
    "_scratch/bi_global", xc="PW", spinorb=True, ngridk=(1, 1, 1),
    extra_blocks={"socscf": [3.0]},
).get_energy()

e_per_species = bi_structure.get_calculation(
    "_scratch/bi_per_species", xc="PW", spinorb=True, ngridk=(1, 1, 1),
    soc_scale={"Bi": 3.0},
).get_energy()

print(f"global socscf=3.0:        {e_global:.10f} Hartree")
print(f"per-species soc_scale=3.0: {e_per_species:.10f} Hartree")
print(f"|delta|:                   {abs(e_global - e_per_species):.2e} Hartree")

global socscf=3.0:        -21569.7401469560 Hartree
per-species soc_scale=3.0: -21569.7401469560 Hartree
|delta|:                   0.00e+00 Hartree


## Independent per-species scaling

Now a two-species cell far enough apart to converge independently: bismuth (Z = 83,
strong SOC) and silicon (Z = 14, weak SOC). Turning off each species' SOC scale in
turn should move the total energy away from the `spinorb=True` default -- and the
bismuth effect should dwarf the silicon effect, confirming the scale really is
applied per-atom-via-species, not globally.

In [3]:
BISI_AVEC = [(10.0, 0.0, 0.0), (0.0, 10.0, 0.0), (0.0, 0.0, 10.0)]
bisi_structure = Structure(BISI_AVEC, {"Bi": [(0.0, 0.0, 0.0)], "Si": [(0.5, 0.5, 0.5)]})


def energy(label, soc_scale):
    calc = bisi_structure.get_calculation(
        f"_scratch/{label}", xc="PW", spinorb=True, ngridk=(1, 1, 1), soc_scale=soc_scale
    )
    return calc.get_energy()


e_default = energy("bisi_default", None)
e_bi_off = energy("bisi_bi_off", {"Bi": 0.0})
e_si_off = energy("bisi_si_off", {"Si": 0.0})

delta_bi = abs(e_default - e_bi_off)
delta_si = abs(e_default - e_si_off)

print(f"default (spinorb=True):      {e_default:.8f} Hartree")
print(f"Bi soc_scale=0.0:            {e_bi_off:.8f} Hartree  (|delta| = {delta_bi:.2e})")
print(f"Si soc_scale=0.0:            {e_si_off:.8f} Hartree  (|delta| = {delta_si:.2e})")
print(f"\nturning off Bi's SOC moves the energy {delta_bi / delta_si:.0f}x more than turning off Si's")

default (spinorb=True):      -21858.82316865 Hartree
Bi soc_scale=0.0:            -21858.79191541 Hartree  (|delta| = 3.13e-02)
Si soc_scale=0.0:            -21858.82371096 Hartree  (|delta| = 5.42e-04)

turning off Bi's SOC moves the energy 58x more than turning off Si's


## Next

`05_berry_curvature.ipynb` -- Berry curvature and Chern numbers via a Wilson-loop
method (also a new Fortran extension, not upstream Elk).